<a href="https://colab.research.google.com/github/Melonangie/Galaxy-Vision/blob/main/MaNGA_EDA_morphology_spectra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔭 EDA of MaNGA Galaxies: Morphology Metadata + Integrated Spectra

This notebook runs a general **exploratory data analysis (EDA)** on two inputs stored in Google Drive:

| Input | Format | Content |
|---|---|---|
| `morfologia.fits` | FITS table, ~16k rows (one per galaxy) | `name, objra, objdec, Type, Ttype, nsa_z, LogMass, incl, PETRO_TH90` |
| `spectra.zip` | ~16k `<name>.fits.gz` files | `WAVE, FLUX_1RE, FLUX_FoV, ERROR_1RE, ERROR_FoV` |

Each spectrum file is named after the galaxy `name` in the metadata (e.g. `manga-10001-1901.fits.gz`).

**Things already seen in the sample CSV exports (handled below):**
* `WAVE` runs from ~3622 to ~10351 Å in **1.5 Å steps**, but blocks of pixels were removed, so spectra have **gaps** and **different lengths**.
* Some trailing rows have **zero flux** and placeholder errors (e.g. `4.0`, `34.0`). These are treated as padding and masked.
* Metadata uses `-9999` and empty cells for **missing values** (e.g. `nsa_z`, `LogMass`).
* `Ttype`: −5 = E, −2 = S0, 0 = S0a, 1…7 = Sa…Sd, and **11** for the generic class `S`.
* `ERROR_FoV` looks almost constant inside each file and is larger than `FLUX_FoV`, so check what it means before using it as a weight.

**Assumptions** (check them in §3.4): `WAVE` is in the **observed frame** (as in the MaNGA DRP), so it is de-redshifted with `nsa_z`. Flux is in MaNGA units (10⁻¹⁷ erg s⁻¹ cm⁻² Å⁻¹).

### Outline
0. Setup  
1. Metadata EDA  
2. Spectra inventory & cross-match  
3. Individual spectra: structure, masking, gaps, frame check  
4. Bulk processing: QC stats + common rest-frame grid (cached)  
5. Spectral quality EDA  
6. Spectral features (D4000, Hδ, Mg b, EW Hα, EW [OII])  
7. **Early vs late type: how spectra and metadata relate**  
8. **How to display spectra grouped by Type: suggestions + code**  
9. Save outputs

## 0. Setup

In [ ]:
# Colab already has numpy, pandas, scipy, scikit-learn, seaborn, plotly and astropy.
# This line only makes sure astropy is recent (FITS I/O, units, coordinates, cosmology).
!pip install -q --upgrade astropy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 87.4 MB/s eta 0:00:00


In [3]:
# ---------- Standard library ----------
import os, re, glob, zipfile, warnings

# ---------- Scientific stack ----------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm
from joblib import Parallel, delayed          # parallel reading of the 16k files

# ---------- Astronomy ----------
from astropy.io import fits                    # FITS reader (handles .fits.gz transparently)
from astropy.table import Table                # FITS table -> pandas
from astropy.coordinates import SkyCoord       # sky map
from astropy.cosmology import Planck18         # angular -> physical sizes
import astropy.units as u

# ---------- Machine learning (section 7) ----------
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier   # handles NaNs natively
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.inspection import permutation_importance

# ---------- Display defaults ----------
from IPython.display import display
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 50)
warnings.filterwarnings('ignore', category=RuntimeWarning)   # e.g. "Mean of empty slice" from nan-functions

In [4]:
# Mount Google Drive (a pop-up asks for authorisation)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================== CONFIGURATION ===========================
# ---- Paths: edit these to match your Drive ----
DRIVE_DIR  = '/content/drive/MyDrive/Proyecto_morfologia/data'                  # folder with the data
META_FILE  = os.path.join(DRIVE_DIR, 'morfologia.fits')      # metadata table
SPEC_ZIP   = os.path.join(DRIVE_DIR, 'MPL-11.zip')          # zip with the ~16k *.fits.gz spectra
LOCAL_SPEC = '/content/spectra'                              # unzip to local VM disk (much faster than Drive)
CACHE_DIR  = os.path.join(DRIVE_DIR, 'eda_cache')            # processed products are cached here
os.makedirs(CACHE_DIR, exist_ok=True)

# ---- Spectral settings ----
SPEC_COLS = ['WAVE', 'FLUX_1RE', 'FLUX_FoV', 'ERROR_1RE', 'ERROR_FoV']
FLUX_COL, ERR_COL = 'FLUX_1RE', 'ERROR_1RE'   # aperture used for the analysis (1 effective radius)
WAVE_IS_OBSERVED = True                       # True -> de-redshift with nsa_z (checked in §3.4)
NATIVE_STEP = 1.5                             # Å, native pixel size in the files
WAVE_FULL = (3622.0, 10351.0)                 # Å, full observed range -> 4487 pixels if nothing were removed
REST_GRID = np.arange(3650.0, 8800.0, 1.5)    # common rest-frame grid covered by almost all galaxies (z <= 0.15)
NORM_WINDOW = (5450, 5550)                    # featureless window: every spectrum is divided by its median here
SNR_MIN = 3                                   # min median S/N per pixel to use a spectrum in stacks / ML
N_JOBS = -1                                   # joblib: use all CPU cores
RANDOM_STATE = 42

# ---- Morphology settings ----
T11_IS_LATE = True                            # Ttype = 11 (generic "S") -> Late; False -> Unknown
CLASS_COLORS = {'Early': '#c0392b', 'Late': '#2e86c1', 'Unknown': '#95a5a6'}
CLASSES = ['Early', 'Late']

# Rest-frame wavelengths (Å) of key spectral features, marked on plots
LINES = {'[OII]': 3727.4, 'K': 3933.7, 'H': 3968.5, 'Hδ': 4101.7, 'G': 4304.4,
         'Hβ': 4861.3, '[OIII]': 5006.8, 'Mg b': 5175.4, 'NaD': 5892.9,
         'Hα': 6562.8, '[NII]': 6583.5, '[SII]': 6724.0}

In [ ]:
# Unzip the spectra once per Colab session to the local disk
if not glob.glob(os.path.join(LOCAL_SPEC, '**', '*.fits*'), recursive=True):
    os.makedirs(LOCAL_SPEC, exist_ok=True)
    with zipfile.ZipFile(SPEC_ZIP) as zf:
        members = [m for m in zf.namelist()
                   if m.endswith(('.fits', '.fits.gz')) and '__MACOSX' not in m]
        for m in tqdm(members, desc='Unzipping'):
            zf.extract(m, LOCAL_SPEC)

# Recursive glob in case the zip contains sub-folders
spec_files = sorted(f for f in glob.glob(os.path.join(LOCAL_SPEC, '**', '*.fits*'), recursive=True)
                    if '__MACOSX' not in f)
print(f'{len(spec_files):,} spectral files found')

## 1. Metadata EDA
### 1.1 Load the FITS table

In [ ]:
def load_fits_table(path):
    """Read the first table HDU of a FITS file into pandas and decode byte strings."""
    df = Table.read(path).to_pandas()
    for col in df.select_dtypes(include='object'):
        df[col] = df[col].map(lambda v: v.decode().strip() if isinstance(v, bytes) else v)
    return df

meta_raw = load_fits_table(META_FILE)
print('Shape:', meta_raw.shape)
meta_raw.head()

### 1.2 Structure and summary statistics

In [ ]:
meta_raw.info()
display(meta_raw.describe(include='all').T)

### 1.3 Cleaning: sentinels, missing values, sanity checks
`-9999` means "missing" and is replaced with `NaN`. Also check duplicates and values that are physically suspicious.

In [ ]:
meta = meta_raw.copy()
meta['name'] = meta['name'].astype(str).str.strip()
meta['Type'] = meta['Type'].astype(str).str.strip().replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

NUM_COLS = ['objra', 'objdec', 'Ttype', 'nsa_z', 'LogMass', 'incl', 'PETRO_TH90']
meta[NUM_COLS] = meta[NUM_COLS].apply(pd.to_numeric, errors='coerce')   # blanks -> NaN
meta[NUM_COLS] = meta[NUM_COLS].mask(meta[NUM_COLS] <= -999)           # -9999 sentinels -> NaN

print('Duplicated names:', meta['name'].duplicated().sum())

# Missing values per column
missing = meta.isna().sum().to_frame('n_missing')
missing['pct'] = (100 * missing['n_missing'] / len(meta)).round(2)
display(missing[missing['n_missing'] > 0])

# Simple physical sanity checks (counts of suspicious rows)
checks = {
    'nsa_z <= 0':                        (meta['nsa_z'] <= 0).sum(),
    'nsa_z > 0.2':                       (meta['nsa_z'] > 0.2).sum(),
    'LogMass < 8 or > 12.5':             ((meta['LogMass'] < 8) | (meta['LogMass'] > 12.5)).sum(),
    'incl == 0 (face-on or a default?)': (meta['incl'] == 0).sum(),
    'incl > 90':                         (meta['incl'] > 90).sum(),
    'PETRO_TH90 <= 0':                   (meta['PETRO_TH90'] <= 0).sum(),
}
display(pd.Series(checks, name='n_rows').to_frame())

### 1.4 Derived columns
* `morph_class`: **Early** (Ttype ≤ 0: E, S0, S0a) vs **Late** (Ttype > 0: spirals and irregulars).
* `group`: coarse Hubble groups for plotting.
* `bar`: bar flag from the `Type` prefix (`SB` = strong, `SAB` = weak).
* `R90_kpc`: physical Petrosian 90% radius from `PETRO_TH90` (arcsec) and `nsa_z` (Planck18 cosmology).

In [ ]:
def morph_class(t):
    """Early (T<=0) / Late (T>0) split; T=11 handled by T11_IS_LATE."""
    if pd.isna(t):
        return 'Unknown'
    if t == 11:
        return 'Late' if T11_IS_LATE else 'Unknown'
    return 'Early' if t <= 0 else 'Late'

meta['morph_class'] = meta['Ttype'].map(morph_class)

# Coarse Hubble groups (right-closed bins): (-inf,-3]=E, (-3,0]=S0-S0a, (0,3]=Sa-Sb, ...
GROUP_ORDER = ['E', 'S0–S0a', 'Sa–Sb', 'Sbc–Sd', 'Sdm–Irr', 'S (T=11)']
meta['group'] = pd.cut(meta['Ttype'], bins=[-np.inf, -3, 0, 3, 7, 10, np.inf], labels=GROUP_ORDER)

# Bar flag from the Type prefix
meta['bar'] = np.select([meta['Type'].str.startswith('SAB', na=False),
                         meta['Type'].str.startswith('SB',  na=False)],
                        ['SAB (weak)', 'SB (strong)'], default='none')

# Physical size: arcsec -> kpc with Planck18 (only where z > 0)
z_safe = meta['nsa_z'].where(meta['nsa_z'] > 0)
kpc_per_arcsec = Planck18.kpc_proper_per_arcmin(z_safe.fillna(0.01).values).value / 60.0
meta['R90_kpc'] = np.where(z_safe.notna(), meta['PETRO_TH90'] * kpc_per_arcsec, np.nan)

# Order Types along the Hubble sequence (by Ttype, then name)
TYPE_ORDER = (meta.groupby('Type')['Ttype'].median().reset_index()
                  .sort_values(['Ttype', 'Type'])['Type'].tolist())

display(pd.crosstab(meta['group'], meta['morph_class'], margins=True))

### 1.5 Categorical distributions: Type, Ttype, bars

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8), gridspec_kw={'width_ratios': [3, 1.4, 1.2]})

# Counts per Type, ordered along the Hubble sequence and coloured by class
sns.countplot(data=meta, x='Type', order=TYPE_ORDER, hue='morph_class',
              palette=CLASS_COLORS, dodge=False, ax=axes[0])
axes[0].tick_params(axis='x', rotation=90); axes[0].set_title('Galaxies per Type')

# Counts per Ttype
meta['Ttype'].value_counts().sort_index().plot.bar(ax=axes[1], color='grey')
axes[1].set_title('Galaxies per Ttype'); axes[1].set_xlabel('Ttype')

# Bar fraction per class
(pd.crosstab(meta['morph_class'], meta['bar'], normalize='index')
   .plot.bar(stacked=True, ax=axes[2], colormap='Set2'))
axes[2].set_title('Bar fraction by class'); axes[2].set_ylabel('fraction'); axes[2].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

### 1.6 Numerical distributions by class and Hubble group

In [ ]:
PLOT_COLS = ['nsa_z', 'LogMass', 'incl', 'PETRO_TH90', 'R90_kpc']
fig, axes = plt.subplots(2, len(PLOT_COLS), figsize=(4.2 * len(PLOT_COLS), 8))

for i, col in enumerate(PLOT_COLS):
    # Top row: normalised histograms Early vs Late
    sns.histplot(data=meta, x=col, hue='morph_class', hue_order=CLASSES, palette=CLASS_COLORS,
                 element='step', stat='density', common_norm=False, ax=axes[0, i])
    # Bottom row: distribution per Hubble group
    sns.boxplot(data=meta, x='group', y=col, order=GROUP_ORDER, color='lightgrey',
                showfliers=False, ax=axes[1, i])
    axes[1, i].tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.show()

### 1.7 Sky distribution

In [ ]:
ok = meta[['objra', 'objdec']].notna().all(axis=1)
coords = SkyCoord(ra=meta.loc[ok, 'objra'].values * u.deg, dec=meta.loc[ok, 'objdec'].values * u.deg)

fig = plt.figure(figsize=(11, 5.5))
ax = fig.add_subplot(111, projection='mollweide')
for cls in ['Late', 'Early']:                       # Early on top
    sel = (meta.loc[ok, 'morph_class'] == cls).values
    ax.scatter(coords.ra.wrap_at(180 * u.deg).radian[sel], coords.dec.radian[sel],
               s=2, alpha=0.5, color=CLASS_COLORS[cls], label=cls)
ax.grid(True); ax.legend(loc='lower right', markerscale=5)
ax.set_title('Sky positions (RA wrapped at 180°)')
plt.show()

### 1.8 Relationships between metadata variables
Mass vs redshift shows the survey **selection** (more massive galaxies are seen at higher z). This matters later because early types are usually more massive.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Spearman correlation (robust to outliers and non-linear trends)
corr = meta[['Ttype', 'nsa_z', 'LogMass', 'incl', 'PETRO_TH90', 'R90_kpc']].corr(method='spearman')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('Spearman correlation')

# Mass vs redshift, coloured by class
sns.scatterplot(data=meta, x='nsa_z', y='LogMass', hue='morph_class', hue_order=CLASSES,
                palette=CLASS_COLORS, s=6, alpha=0.4, linewidth=0, ax=axes[1])
axes[1].set_title('Stellar mass vs redshift')
plt.tight_layout(); plt.show()

# Pair plot on a random subsample (fast enough for 16k rows)
sample = meta[meta['morph_class'].isin(CLASSES)].sample(min(3000, len(meta)), random_state=RANDOM_STATE)
sns.pairplot(sample, vars=['LogMass', 'nsa_z', 'incl', 'R90_kpc'], hue='morph_class',
             palette=CLASS_COLORS, corner=True, plot_kws=dict(s=6, alpha=0.4, linewidth=0))
plt.show()

## 2. Spectra inventory and cross-match with the metadata

In [ ]:
spec_index = pd.DataFrame({'path': spec_files})
# "manga-10001-1901.fits.gz" -> "manga-10001-1901"
spec_index['name'] = spec_index['path'].map(lambda p: re.sub(r'\.fits(\.gz)?$', '', os.path.basename(p)))
spec_index['size_kb'] = spec_index['path'].map(os.path.getsize) / 1024

names_meta, names_spec = set(meta['name']), set(spec_index['name'])
print(f'Metadata rows          : {len(names_meta):,}')
print(f'Spectral files         : {len(names_spec):,}')
print(f'Matched                : {len(names_meta & names_spec):,}')
print(f'Metadata without file  : {len(names_meta - names_spec):,}  e.g. {sorted(names_meta - names_spec)[:5]}')
print(f'Files without metadata : {len(names_spec - names_meta):,}  e.g. {sorted(names_spec - names_meta)[:5]}')
print(f'Duplicated file names  : {spec_index["name"].duplicated().sum()}')

# Keep matched files only, one per galaxy
spec_index = spec_index[spec_index['name'].isin(names_meta)].drop_duplicates('name').reset_index(drop=True)
PATH_OF = dict(zip(spec_index['name'], spec_index['path']))

# File size check: very small files are often empty or corrupted
ax = spec_index['size_kb'].plot.hist(bins=60, figsize=(7, 3.5), title='Spectrum file size (kB)')
plt.show()

## 3. Individual spectra: structure, masking, gaps
### 3.1 Reader and helpers

In [ ]:
def read_spectrum(path):
    """Read one spectrum (first table HDU) into a DataFrame with SPEC_COLS as float64.
    - np.asarray(..., 'f8') also converts FITS big-endian data to native byte order
    - .ravel() handles both layouts: one row per pixel OR one row with array columns
    """
    with fits.open(path, memmap=False) as hdul:
        hdu = next(h for h in hdul if isinstance(h, (fits.BinTableHDU, fits.TableHDU)))
        return pd.DataFrame({c: np.asarray(hdu.data[c], dtype='f8').ravel() for c in SPEC_COLS})


def good_pixels(s, flux_col=FLUX_COL, err_col=ERR_COL):
    """Usable pixels: finite values, positive error, non-zero flux (exact zeros are padding)."""
    f, e = s[flux_col].to_numpy(), s[err_col].to_numpy()
    return np.isfinite(f) & np.isfinite(e) & (e > 0) & (f != 0)


def find_gaps(wave, step=NATIVE_STEP):
    """(start, end) wavelength pairs where pixels are missing from the file."""
    d = np.diff(wave)
    i = np.flatnonzero(d > 1.5 * step)
    return list(zip(wave[i], wave[i + 1]))


def break_at_gaps(w, *ys, step=NATIVE_STEP):
    """Insert NaN at gaps so matplotlib does not draw lines across missing pixels."""
    i = np.flatnonzero(np.diff(w) > 1.5 * step) + 1
    return [np.insert(np.asarray(a, dtype=float), i, np.nan) for a in (w, *ys)]


# FITS structure of one file
with fits.open(spec_index['path'].iloc[0]) as hdul:
    hdul.info()
    print(repr(hdul[1].header))

### 3.2 Raw content of one file

In [ ]:
example = spec_index['name'].iloc[0]
s = read_spectrum(PATH_OF[example])
m = good_pixels(s)

print(f'{example}: {len(s)} rows | good pixels: {m.sum()} | padding/zero rows: {(s[FLUX_COL] == 0).sum()}')
print(f'Max possible rows on a 1.5 Å grid: {int((s.WAVE.max() - s.WAVE.min()) / NATIVE_STEP) + 1}')
print('Gaps (Å):', [(round(a, 1), round(b, 1)) for a, b in find_gaps(s['WAVE'].to_numpy())][:10], '...')
display(s.describe().T)

### 3.3 Plot an early-type and a late-type spectrum
Black = `FLUX_1RE` with a ±1σ band. Orange (right axis) = `FLUX_FoV`. Grey bands = missing pixels. Dotted lines = features at their **observed** wavelengths.

In [ ]:
META_IDX = meta.set_index('name')

def plot_spectrum(name, ax=None, show_fov=True, show_lines=True):
    """Observed-frame plot of one galaxy: 1RE flux + error, FoV flux, gaps and line markers."""
    row = META_IDX.loc[name]
    s = read_spectrum(PATH_OF[name]); m = good_pixels(s)
    w, f, e = break_at_gaps(s['WAVE'][m].to_numpy(), s['FLUX_1RE'][m], s['ERROR_1RE'][m])
    z = row['nsa_z'] if (WAVE_IS_OBSERVED and pd.notna(row['nsa_z'])) else 0.0
    ax = ax or plt.subplots(figsize=(15, 4))[1]

    ax.plot(w, f, lw=0.6, color='k', label='FLUX_1RE')
    ax.fill_between(w, f - e, f + e, color='k', alpha=0.2, lw=0)
    ax.set_ylim(np.nanpercentile(f, 0.5) * 0.8, np.nanpercentile(f, 99.5) * 1.3)   # robust y-range
    if show_fov:
        mf = good_pixels(s, 'FLUX_FoV', 'ERROR_FoV')
        ax2 = ax.twinx(); ax2.grid(False)
        ax2.plot(*break_at_gaps(s['WAVE'][mf].to_numpy(), s['FLUX_FoV'][mf]), lw=0.5, color='tab:orange', alpha=0.6)
        ax2.set_ylabel('FLUX_FoV', color='tab:orange')
    for a, b in find_gaps(s['WAVE'].to_numpy()):
        ax.axvspan(a, b, color='grey', alpha=0.15, lw=0)
    if show_lines:
        for lab, lam in LINES.items():
            ax.axvline(lam * (1 + z), ls=':', lw=0.7, color='tab:green')
            ax.text(lam * (1 + z), 0.98, lab, transform=ax.get_xaxis_transform(),
                    rotation=90, fontsize=7, va='top', ha='right')
    ax.set_xlabel('Observed wavelength [Å]'); ax.set_ylabel('FLUX_1RE')
    ax.set_title(f"{name} | Type={row['Type']} (T={row['Ttype']:g}) | z={row['nsa_z']:.4f} | "
                 f"logM*={row['LogMass']:.2f} | incl={row['incl']:.0f}°")
    return ax

# One early-type and one late-type example with a known redshift
has_file = meta['name'].isin(PATH_OF) & meta['nsa_z'].notna()
ex_early = meta[has_file & (meta['morph_class'] == 'Early')].sample(1, random_state=1)['name'].iloc[0]
ex_late  = meta[has_file & (meta['morph_class'] == 'Late')].sample(1, random_state=1)['name'].iloc[0]
fig, axes = plt.subplots(2, 1, figsize=(15, 8))
plot_spectrum(ex_early, ax=axes[0]); plot_spectrum(ex_late, ax=axes[1])
plt.tight_layout(); plt.show()

### 3.4 Check: is `WAVE` in the observed frame or the rest frame?
For late-type galaxies with z > 0.02, compare the Hα peak contrast at **6563·(1+z)** (observed frame) with the contrast at **6563 Å** (rest frame). If the observed position wins most of the time, keep `WAVE_IS_OBSERVED = True`.

In [ ]:
def peak_contrast(w, f, center, half=8, cont_half=60):
    """Max flux within ±half Å of `center` divided by the median flux of the surrounding window."""
    line = np.abs(w - center) <= half
    cont = (np.abs(w - center) <= cont_half) & ~line
    if line.sum() < 3 or cont.sum() < 10:
        return np.nan
    return np.nanmax(f[line]) / np.nanmedian(f[cont])

cand = meta[has_file & (meta['morph_class'] == 'Late') & (meta['nsa_z'] > 0.02)]
cand = cand.sample(min(150, len(cand)), random_state=RANDOM_STATE)
res = []
for name, z in zip(cand['name'], cand['nsa_z']):
    s = read_spectrum(PATH_OF[name]); m = good_pixels(s)
    w, f = s['WAVE'].to_numpy()[m], s[FLUX_COL].to_numpy()[m]
    res.append((peak_contrast(w, f, 6562.8 * (1 + z)), peak_contrast(w, f, 6562.8)))
res = np.array(res, dtype=float)
valid = np.isfinite(res).all(axis=1)
print(f'Hα stronger at the OBSERVED-frame position in {np.mean(res[valid, 0] > res[valid, 1]):.0%} '
      f'of {valid.sum()} late-type galaxies  ->  WAVE_IS_OBSERVED should be '
      f'{np.mean(res[valid, 0] > res[valid, 1]) > 0.5}')

## 4. Bulk processing: QC stats + common rest-frame grid
A single pass over all files:
1. reads the file and masks bad pixels,
2. computes **QC statistics**: `completeness` (good pixels / 4487 possible), gaps, S/N, negative-flux fraction, …,
3. de-redshifts, **resamples onto `REST_GRID`** (gaps stay `NaN`, no interpolation across them), and **normalises** by the median flux in `NORM_WINDOW`.

The result is a matrix `SPEC` (n_galaxies × n_wavelengths, float32, ~200–250 MB for 16k galaxies). It is **cached on Drive**, so later sessions skip this step. Delete the cache files if you change the settings.

In [ ]:
def resample(w, f, grid, max_gap=2.5 * NATIVE_STEP):
    """Linear interpolation onto `grid`. Points outside coverage or inside gaps wider than max_gap -> NaN."""
    out = np.interp(grid, w, f, left=np.nan, right=np.nan)
    j = np.clip(np.searchsorted(w, grid), 1, len(w) - 1)
    out[(w[j] - w[j - 1]) > max_gap] = np.nan
    return out


def process_one(path, z, flux_col=FLUX_COL, err_col=ERR_COL, grid=REST_GRID):
    """Read one file -> (QC dict, normalised rest-frame spectrum on `grid`)."""
    qc = {'error': '', 'norm': np.nan}
    spec = np.full(grid.size, np.nan, dtype='f4')
    try:
        s = read_spectrum(path)
        wave = s['WAVE'].to_numpy()
        m = good_pixels(s, flux_col, err_col)
        w, f, e = wave[m], s[flux_col].to_numpy()[m], s[err_col].to_numpy()[m]
        mf = good_pixels(s, 'FLUX_FoV', 'ERROR_FoV')
        gaps = find_gaps(wave)
        n_full = int(round((WAVE_FULL[1] - WAVE_FULL[0]) / NATIVE_STEP)) + 1
        qc.update(n_rows=len(s), n_good=int(m.sum()), frac_good=m.mean(), completeness=m.sum() / n_full,
                  wave_min=w.min(), wave_max=w.max(),
                  n_gaps=len(gaps), max_gap=max([b - a for a, b in gaps], default=0.0),
                  median_flux=np.median(f), median_err=np.median(e),
                  median_snr=np.median(f / e), frac_neg=np.mean(f < 0),
                  median_flux_fov=np.median(s['FLUX_FoV'].to_numpy()[mf]),
                  median_snr_fov=np.median(s['FLUX_FoV'].to_numpy()[mf] / s['ERROR_FoV'].to_numpy()[mf]))

        zz = z if WAVE_IS_OBSERVED else 0.0
        if np.isfinite(zz) and len(w) > 100:
            spec = resample(w / (1 + zz), f, grid, max_gap=2.5 * NATIVE_STEP / (1 + zz))
            win = (grid >= NORM_WINDOW[0]) & (grid <= NORM_WINDOW[1])
            qc['norm'] = norm = np.nanmedian(spec[win])
            spec = (spec / norm).astype('f4') if norm > 0 else np.full(grid.size, np.nan, 'f4')
    except Exception as ex:                       # keep going, but record the problem
        qc['error'] = f'{type(ex).__name__}: {ex}'
    return qc, spec


CACHE_QC   = os.path.join(CACHE_DIR, f'qc_{FLUX_COL}.parquet')
CACHE_SPEC = os.path.join(CACHE_DIR, f'rest_spectra_{FLUX_COL}.npy')

targets = spec_index[['name', 'path']].merge(meta[['name', 'nsa_z']], on='name', how='left')

if os.path.exists(CACHE_QC) and os.path.exists(CACHE_SPEC):
    qc = pd.read_parquet(CACHE_QC); SPEC = np.load(CACHE_SPEC)
    print('Loaded from cache')
else:
    results = Parallel(n_jobs=N_JOBS, batch_size=32)(
        delayed(process_one)(p, z) for p, z in tqdm(zip(targets['path'], targets['nsa_z']),
                                                    total=len(targets), desc='Processing spectra'))
    qc = pd.DataFrame([r[0] for r in results])
    qc.insert(0, 'name', targets['name'].values)
    SPEC = np.vstack([r[1] for r in results]).astype('f4')
    qc.to_parquet(CACHE_QC); np.save(CACHE_SPEC, SPEC)

assert len(qc) == SPEC.shape[0]
print(f'SPEC matrix: {SPEC.shape}, {SPEC.nbytes / 1e6:.0f} MB')

# Master table: row i of `df` <-> row i of `SPEC` (left merge keeps the order)
df = qc.merge(meta, on='name', how='left')

## 5. Spectral quality EDA

In [ ]:
print('Files with read errors:', (df['error'] != '').sum())
display(df.loc[df['error'] != '', ['name', 'error']].head())

df['log_snr'] = np.log10(df['median_snr'].clip(lower=1e-3))
QC_COLS = ['n_good', 'completeness', 'wave_min', 'wave_max', 'n_gaps', 'max_gap', 'log_snr', 'frac_neg']
display(df[QC_COLS + ['median_flux', 'median_err', 'median_snr_fov']].describe().T)

fig, axes = plt.subplots(2, 4, figsize=(19, 7))
for ax, col in zip(axes.ravel(), QC_COLS):
    sns.histplot(data=df, x=col, hue='morph_class', hue_order=CLASSES, palette=CLASS_COLORS,
                 element='step', bins=50, ax=ax)
plt.tight_layout(); plt.show()

**1RE vs FoV noise.** In the sample files, `ERROR_FoV` was nearly constant and larger than `FLUX_FoV`. The plot below shows whether that holds for the whole sample.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# S/N of the two apertures
axes[0].scatter(df['median_snr'], df['median_snr_fov'], s=3, alpha=0.3)
axes[0].set(xscale='log', yscale='log', xlabel='median S/N (1RE)', ylabel='median S/N (FoV)',
            title='S/N per pixel: 1RE vs FoV')

# Fraction of galaxies with valid data at each rest wavelength
coverage = np.isfinite(SPEC).mean(axis=0)
axes[1].plot(REST_GRID, coverage, color='k', lw=0.8)
axes[1].set(xlabel='Rest wavelength [Å]', ylabel='fraction of galaxies', title='Rest-frame coverage', ylim=(0, 1.02))

# S/N vs stellar mass
sns.scatterplot(data=df, x='LogMass', y='log_snr', hue='morph_class', hue_order=CLASSES,
                palette=CLASS_COLORS, s=5, alpha=0.4, linewidth=0, ax=axes[2])
axes[2].axhline(np.log10(SNR_MIN), ls='--', color='k'); axes[2].set_title('S/N vs mass (dashed = SNR_MIN)')
plt.tight_layout(); plt.show()

In [ ]:
# Quality flag used for stacks, PCA and ML from here on
df['use'] = ((df['error'] == '') & (df['median_snr'] >= SNR_MIN) & (df['norm'] > 0)
             & df['morph_class'].isin(CLASSES))
print(df['use'].value_counts())
display(pd.crosstab(df['group'], df['use'], margins=True))

# The lowest-S/N spectra are worth looking at by eye
worst = df[df['error'] == ''].nsmallest(3, 'median_snr')['name']
fig, axes = plt.subplots(len(worst), 1, figsize=(15, 3.2 * len(worst)))
for ax, n in zip(np.atleast_1d(axes), worst):
    plot_spectrum(n, ax=ax, show_fov=False, show_lines=False)
plt.tight_layout(); plt.show()

## 6. Spectral features (indices)
These are measured on the rest-frame matrix and vectorised over all galaxies. They are ratios or equivalent widths (EW), so the normalisation does not affect them.

| Index | Traces | Early types | Late types |
|---|---|---|---|
| **D4000n** (Balogh+99) | age of the stellar population | high (≳1.6) | low (≲1.5) |
| **HδA** (Worthey & Ottaviani 97) | recent star formation (0.1–1 Gyr) | low or negative | high |
| **Mg b** (Lick) | old, metal-rich stars | strong | weaker |
| **EW(Hα)**, **EW([OII])** (emission > 0) | current star formation or ionised gas | ~0 (or LINER-like) | large |
| **color_RB** = ⟨F 6000–6200⟩/⟨F 4400–4600⟩ | continuum colour | red (high) | blue (low) |

In [ ]:
G, dG = REST_GRID, REST_GRID[1] - REST_GRID[0]

def band(lo, hi):
    return (G >= lo) & (G <= hi)

def mean_flux(S, lo, hi):
    return np.nanmean(S[:, band(lo, hi)], axis=1)

def d4000n(S):
    """Narrow 4000 Å break: <F_nu>(4000-4100) / <F_nu>(3850-3950), with F_nu ∝ F_lambda·λ²."""
    r, b = band(4000, 4100), band(3850, 3950)
    return np.nanmean(S[:, r] * G[r] ** 2, axis=1) / np.nanmean(S[:, b] * G[b] ** 2, axis=1)

def equivalent_width(S, line, blue, red, min_valid=0.8):
    """EW (Å) with a linear pseudo-continuum through the two side-bands. Positive = absorption."""
    cb, cr = mean_flux(S, *blue), mean_flux(S, *red)
    lb, lr = np.mean(blue), np.mean(red)
    m = band(*line); w = G[m]
    cont = cb[:, None] + (cr - cb)[:, None] * (w - lb) / (lr - lb)
    depth = 1 - S[:, m] / cont
    ew = np.nanmean(depth, axis=1) * (w.size * dG)          # mean depth × band width
    return np.where(np.isfinite(depth).mean(axis=1) >= min_valid, ew, np.nan)

# name: (feature band, blue continuum, red continuum, sign)   sign=-1 -> emission EW positive
INDEX_DEFS = {
    'HdA':    ((4083.500, 4122.250), (4041.600, 4079.750), (4128.500, 4161.000), +1),
    'Mgb':    ((5160.125, 5192.625), (5142.625, 5161.375), (5191.375, 5206.375), +1),
    'EW_OII': ((3716.0, 3738.0),     (3653.0, 3713.0),     (3741.0, 3801.0),     -1),
    'EW_Ha':  ((6553.0, 6573.0),     (6490.0, 6530.0),     (6600.0, 6640.0),     -1),
}

df['D4000n'] = d4000n(SPEC)
for name, (line, blue, red, sign) in INDEX_DEFS.items():
    df[name] = sign * equivalent_width(SPEC, line, blue, red)
df['color_RB'] = mean_flux(SPEC, 6000, 6200) / mean_flux(SPEC, 4400, 4600)
df['log_EW_Ha'] = np.log10(df['EW_Ha'].clip(lower=0.1))     # log scale for plots; floor at 0.1 Å

INDEX_COLS = ['D4000n', 'HdA', 'Mgb', 'EW_OII', 'EW_Ha', 'color_RB']
display(df.loc[df['use'], INDEX_COLS].describe(percentiles=[.01, .5, .99]).T)

In [ ]:
fig, axes = plt.subplots(1, len(INDEX_COLS), figsize=(4 * len(INDEX_COLS), 3.8))
for ax, col in zip(axes, INDEX_COLS):
    d = df[df['use']]
    lo, hi = d[col].quantile([0.01, 0.99])                  # clip extreme tails for display
    sns.histplot(data=d[d[col].between(lo, hi)], x=col, hue='morph_class', hue_order=CLASSES,
                 palette=CLASS_COLORS, element='step', stat='density', common_norm=False, ax=ax)
plt.tight_layout(); plt.show()

## 7. Early vs late type: how spectra and metadata relate
The question: **how well do the metadata and the spectra separate early types (E/S0/S0a) from late types (spirals/irregulars), and which quantities drive the split?**

### 7.1 Univariate comparison (effect sizes, not only p-values)
With ~16k galaxies almost every p-value is tiny. The **KS statistic D** (0 = identical, 1 = fully separated) and the **Spearman ρ with Ttype** are more useful measures.

In [ ]:
d = df[df['use']]
rows = []
for col in ['LogMass', 'nsa_z', 'incl', 'PETRO_TH90', 'R90_kpc'] + INDEX_COLS:
    a = d.loc[d['morph_class'] == 'Early', col].dropna()
    b = d.loc[d['morph_class'] == 'Late',  col].dropna()
    hub = d[d['Ttype'] <= 10]                               # Hubble sequence only (excludes T=11)
    rows.append({'feature': col,
                 'median_Early': a.median(), 'median_Late': b.median(),
                 'KS_D': stats.ks_2samp(a, b).statistic,
                 'MannWhitney_p': stats.mannwhitneyu(a, b).pvalue,
                 'spearman_vs_Ttype': hub[['Ttype', col]].corr(method='spearman').iloc[0, 1]})
sep = pd.DataFrame(rows).set_index('feature').sort_values('KS_D', ascending=False)
display(sep.style.format('{:.3g}').background_gradient(subset=['KS_D'], cmap='Greens'))

### 7.2 Spectral indices along the Hubble sequence

In [ ]:
d = df[df['use'] & (df['Ttype'] <= 10)].copy()
d['Ttype_int'] = d['Ttype'].astype(int)
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
for ax, col in zip(axes.ravel(), ['D4000n', 'log_EW_Ha', 'Mgb', 'HdA']):
    lo, hi = d[col].quantile([0.01, 0.99])
    sns.boxplot(data=d, x='Ttype_int', y=col, hue='morph_class', palette=CLASS_COLORS,
                dodge=False, showfliers=False, ax=ax)
    ax.set_ylim(lo, hi); ax.set_xlabel('Ttype'); ax.legend_.remove()
plt.suptitle('Spectral indices vs Ttype'); plt.tight_layout(); plt.show()

### 7.3 Diagnostic diagrams combining spectra and metadata
* **D4000n vs HδA**: star-formation history plane (Kauffmann+03). Early types are old and quiescent, late types are younger.
* **D4000n vs log EW(Hα)**: quiescent vs star-forming.
* **D4000n vs LogMass**: separates **mass** from **morphology** effects.

In [ ]:
d = df[df['use']]
pairs = [('D4000n', 'HdA'), ('D4000n', 'log_EW_Ha'), ('LogMass', 'D4000n')]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, (x, y) in zip(axes, pairs):
    for cls in ['Late', 'Early']:
        dd = d[d['morph_class'] == cls]
        ax.scatter(dd[x], dd[y], s=3, alpha=0.25, color=CLASS_COLORS[cls], label=cls)
    # density contours make the two populations readable when points overlap
    sns.kdeplot(data=d, x=x, y=y, hue='morph_class', hue_order=CLASSES, palette=CLASS_COLORS,
                levels=5, linewidths=1, ax=ax, legend=False)
    ax.set_xlim(d[x].quantile([0.005, 0.995])); ax.set_ylim(d[y].quantile([0.005, 0.995]))
    ax.set(xlabel=x, ylabel=y)
axes[0].legend(markerscale=5)
plt.tight_layout(); plt.show()

### 7.4 Stacked spectra: Early vs Late
Median of the normalised rest-frame spectra, with a 16–84th percentile band. The bottom panel shows the Early/Late ratio, which highlights where the two classes differ.

In [ ]:
def stack(mask, q=(16, 50, 84), min_n=5):
    """Percentile stack (rows = q) of the normalised rest-frame spectra selected by boolean `mask`."""
    S = SPEC[np.asarray(mask)]
    p = np.nanpercentile(S, q, axis=0)
    p[:, np.isfinite(S).sum(axis=0) < min_n] = np.nan
    return p

def mark_lines(ax, z=0.0, fontsize=7):
    for lab, lam in LINES.items():
        if ax.get_xlim()[0] < lam * (1 + z) < ax.get_xlim()[1]:
            ax.axvline(lam * (1 + z), ls=':', lw=0.6, color='grey')
            ax.text(lam * (1 + z), 0.98, lab, transform=ax.get_xaxis_transform(),
                    rotation=90, fontsize=fontsize, va='top', ha='right', color='grey')

CLASS_STACKS = {c: stack(df['use'] & (df['morph_class'] == c)) for c in CLASSES}

fig, (ax, axr) = plt.subplots(2, 1, figsize=(16, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
for c, p in CLASS_STACKS.items():
    n = (df['use'] & (df['morph_class'] == c)).sum()
    ax.plot(G, p[1], color=CLASS_COLORS[c], lw=0.9, label=f'{c} (n={n:,})')
    ax.fill_between(G, p[0], p[2], color=CLASS_COLORS[c], alpha=0.15, lw=0)
ax.set_ylabel('Normalised flux'); ax.legend(); ax.set_xlim(G[0], G[-1]); mark_lines(ax)
axr.plot(G, CLASS_STACKS['Early'][1] / CLASS_STACKS['Late'][1], color='k', lw=0.7)
axr.axhline(1, ls='--', color='grey'); axr.set_ylabel('Early / Late'); axr.set_xlabel('Rest wavelength [Å]')
plt.tight_layout(); plt.show()

### 7.5 PCA of the spectra: unsupervised structure
If morphology shows up in the spectra, the classes separate along the first principal components. Wavelengths valid in fewer than 60% of galaxies are dropped. The remaining missing pixels are filled with the median spectrum. The files have many gaps, so check the printed matrix size.

In [ ]:
use_idx = np.flatnonzero(df['use'])
X = SPEC[use_idx]
PCA_MIN_COVERAGE = 0.6                                         # keep wavelengths valid in >= 60% of galaxies
cols_ok = np.isfinite(X).mean(axis=0) >= PCA_MIN_COVERAGE
X = X[:, cols_ok]
X = np.where(np.isfinite(X), X, np.nanmedian(X, axis=0))       # fill gaps with the median spectrum
print(f'PCA input: {X.shape[0]} galaxies x {X.shape[1]} wavelengths')

pca = PCA(n_components=min(10, *X.shape), random_state=RANDOM_STATE)
scores = pca.fit_transform(X)
df.loc[use_idx, ['PC1', 'PC2', 'PC3']] = scores[:, :3]

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
axes[0].bar(np.arange(1, pca.n_components_ + 1), pca.explained_variance_ratio_ * 100); axes[0].set(xlabel='PC', ylabel='% variance')
sc = axes[1].scatter(scores[:, 0], scores[:, 1], c=df.loc[use_idx, 'Ttype'].clip(upper=10),
                     cmap='RdYlBu', s=3, alpha=0.5)
plt.colorbar(sc, ax=axes[1], label='Ttype'); axes[1].set(xlabel='PC1', ylabel='PC2', title='PCA scores coloured by Ttype')
for k in range(3):                                             # eigen-spectra
    axes[2].plot(G[cols_ok], pca.components_[k] + 0.1 * k, lw=0.6, label=f'PC{k+1}')
axes[2].legend(); axes[2].set(xlabel='Rest wavelength [Å]', title='Eigen-spectra (offset)')
plt.tight_layout(); plt.show()

### 7.6 How much do metadata and spectra predict Early/Late?
A gradient-boosting classifier (handles NaNs, no scaling needed) with 5-fold CV and **balanced accuracy** (the classes are imbalanced). Three feature sets:
(a) metadata only, (b) spectral features only, (c) both. `nsa_z` is left out because it reflects survey selection more than galaxy physics.

In [ ]:
FEATS = {
    'metadata only': ['LogMass', 'incl', 'R90_kpc'],
    'spectra only':  INDEX_COLS + ['PC1', 'PC2', 'PC3'],
    'metadata + spectra': ['LogMass', 'incl', 'R90_kpc'] + INDEX_COLS + ['PC1', 'PC2', 'PC3'],
}
d = df[df['use']]
y = (d['morph_class'] == 'Early').astype(int)
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

cv_res = {k: cross_val_score(HistGradientBoostingClassifier(random_state=RANDOM_STATE),
                             d[f], y, cv=cv, scoring='balanced_accuracy')
          for k, f in FEATS.items()}
display(pd.DataFrame({k: {'balanced_acc_mean': v.mean(), 'std': v.std()} for k, v in cv_res.items()}).T.round(3))

# Permutation importance for the combined model on a held-out split
f_all = FEATS['metadata + spectra']
Xtr, Xte, ytr, yte = train_test_split(d[f_all], y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
clf = HistGradientBoostingClassifier(random_state=RANDOM_STATE).fit(Xtr, ytr)
imp = permutation_importance(clf, Xte, yte, scoring='balanced_accuracy', n_repeats=10, random_state=RANDOM_STATE)
pd.Series(imp.importances_mean, index=f_all).sort_values().plot.barh(figsize=(7, 5), title='Permutation importance')
plt.show()

### 7.7 How to read these results
What to look for (these are the patterns usually expected; check them against the outputs above):

* **Continuum shape and age.** Early types should have a stronger **D4000n**, a redder `color_RB`, deeper **Ca II H&K**, **G band**, **Mg b** and **NaD**, and weak HδA. Late types should be bluer, with lower D4000n and stronger Balmer absorption. The Early/Late ratio in §7.4 should rise towards the red and peak at the metal lines.
* **Emission lines.** Late types should show strong **Hα, [NII], [OII], Hβ, [OIII]** emission. Some E/S0 galaxies also have weak Hα+[NII] emission, often LINER-like rather than star formation. Use EW(Hα) and, ideally, a BPT diagram before calling an early type "star-forming".
* **Hubble sequence, not two boxes.** The indices in §7.2 should change gradually with Ttype. S0/S0a and Sa galaxies overlap most, which is where classifier errors concentrate.
* **Mass is a confounder.** Early types are on average more massive (§1.6), and D4000 also rises with mass. Compare classes **at fixed mass** (§8.4) before attributing spectral differences to morphology. In §7.6, if *metadata only* already gives high accuracy, mass is doing a lot of the work.
* **Aperture.** `FLUX_1RE` samples the inner galaxy, which for spirals means the bulge. Late types can therefore look older in 1RE than in FoV (§8.7).
* **Inclination.** Late types seen edge-on (high `incl`) are reddened by dust and can look like early types. Restrict to `incl < 60°` for clean late-type stacks.
* **Data quality.** Low-S/N spectra, missing `nsa_z` (no de-redshift possible) and the possibly odd `ERROR_FoV` should be excluded or fixed before drawing conclusions.

## 8. How to display spectra grouped by Type: suggestions
Plotting thousands of individual spectra on top of each other gives an unreadable figure. These views work better:

| # | Visualisation | Shows | Best for |
|---|---|---|---|
| 8.1 | **Waterfall of median stacks per Type** (offset, ordered by Ttype, colour = Ttype) | gradual change along the Hubble sequence | overview figure |
| 8.2 | **Spectral heat-map** (one row per galaxy, sorted by Ttype then mass) + ratio to the global median | every galaxy at once; emission lines appear as vertical stripes | spotting outliers and mis-classifications |
| 8.3 | **Zoom panels** on diagnostic windows (4000 Å break, Hβ+[OIII], Mg b, NaD, Hα+[NII]+[SII]) | subtle features hidden in the full range | papers and talks |
| 8.4 | **Mass-controlled stacks** (Early vs Late in mass bins) | morphology effect at fixed mass | removing the mass confounder |
| 8.5 | **Small-multiples gallery** (random galaxies per group) | spread around the median | honest view of diversity |
| 8.6 | **Interactive** plotly stacks + a widget to browse individual galaxies | exploring by Type | exploratory work in Colab |
| 8.7 | **1RE vs FoV** stacks per class | aperture effect (bulge vs disc) | interpreting late types |
| 8.8 | **Images next to spectra** (optional, Marvin) | morphology ↔ spectrum link | sanity checks |

General tips: always use **rest-frame, normalised** spectra; show **percentile bands** rather than ±σ; keep `NaN` gaps visible (no interpolation across them); use a **diverging colour map** for Ttype (red = early, blue = late); report **n** per stack; for late types consider an `incl < 60°` cut.

### 8.1 Waterfall: median spectrum per Type, ordered by Ttype

In [ ]:
MIN_PER_TYPE = 20                                    # skip Types with too few good spectra
cmap, cnorm = plt.cm.RdYlBu, mcolors.Normalize(vmin=-5, vmax=10)
TYPE_TTYPE = meta.groupby('Type')['Ttype'].median()

types_ok = [t for t in TYPE_ORDER if (df['use'] & (df['Type'] == t)).sum() >= MIN_PER_TYPE]
TYPE_STACKS = {t: stack(df['use'] & (df['Type'] == t)) for t in types_ok}

OFFSET = 0.6
fig, ax = plt.subplots(figsize=(15, 1.0 + 0.55 * len(types_ok)))
for k, t in enumerate(types_ok):
    n = (df['use'] & (df['Type'] == t)).sum()
    ax.plot(G, TYPE_STACKS[t][1] + k * OFFSET, lw=0.8, color=cmap(cnorm(min(TYPE_TTYPE[t], 10))))
    ax.text(G[-1] + 30, 1 + k * OFFSET, f'{t}  (T={TYPE_TTYPE[t]:g}, n={n})', va='center', fontsize=8)
ax.set_xlim(G[0], G[-1] + 700); mark_lines(ax)
ax.set(xlabel='Rest wavelength [Å]', ylabel='Normalised flux + offset', title='Median spectrum per Type')
plt.tight_layout(); plt.show()

### 8.2 Heat-map of all spectra, sorted by Ttype then stellar mass

In [ ]:
order = df[df['use']].sort_values(['Ttype', 'LogMass']).index.to_numpy()
IMG = SPEC[order]
RATIO = IMG / np.nanmedian(IMG, axis=0)              # ratio to the global median highlights differences
t_sorted = df.loc[order, 'Ttype'].to_numpy()
edges = np.flatnonzero(np.diff(t_sorted)) + 1        # rows where Ttype changes

fig, axes = plt.subplots(1, 2, figsize=(19, 9), sharey=True)
for ax, img, title, cm in [(axes[0], IMG, 'Normalised flux', 'magma'),
                           (axes[1], RATIO, 'Flux / global median', 'RdBu_r')]:
    vmin, vmax = np.nanpercentile(img, [2, 98])
    ax.imshow(img, aspect='auto', cmap=cm, vmin=vmin, vmax=vmax, interpolation='nearest',
              extent=[G[0], G[-1], len(img), 0])
    for e in edges:
        ax.axhline(e, color='w', lw=0.5)
    ax.set(title=title, xlabel='Rest wavelength [Å]')
# Label each Ttype block in the middle
mids = (np.r_[0, edges] + np.r_[edges, len(t_sorted)]) / 2
axes[0].set_yticks(mids); axes[0].set_yticklabels([f'T={t:g}' for t in t_sorted[np.r_[0, edges]]])
plt.tight_layout(); plt.show()

### 8.3 Zoom panels on diagnostic windows, one curve per Hubble group

In [ ]:
WINDOWS = {'4000 Å break, Ca H&K, Hδ': (3800, 4200), 'Hβ + [OIII]': (4820, 5030),
           'Mg b': (5120, 5260), 'NaD': (5850, 5940), 'Hα + [NII] + [SII]': (6520, 6760)}
groups_ok = [g for g in GROUP_ORDER if (df['use'] & (df['group'] == g)).sum() >= MIN_PER_TYPE]
GROUP_STACKS = {g: stack(df['use'] & (df['group'] == g)) for g in groups_ok}
gcolors = dict(zip(groups_ok, sns.color_palette('RdYlBu', len(groups_ok))))

fig, axes = plt.subplots(1, len(WINDOWS), figsize=(22, 4.5))
for ax, (title, (lo, hi)) in zip(axes, WINDOWS.items()):
    m = band(lo, hi)
    for g, p in GROUP_STACKS.items():
        ax.plot(G[m], p[1][m], color=gcolors[g], lw=1.1, label=g)
        ax.fill_between(G[m], p[0][m], p[2][m], color=gcolors[g], alpha=0.08, lw=0)
    ax.set(title=title, xlim=(lo, hi), xlabel='Rest λ [Å]'); mark_lines(ax)
axes[0].set_ylabel('Normalised flux'); axes[-1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### 8.4 Mass-controlled stacks: Early vs Late at fixed stellar mass

In [ ]:
MASS_BINS = [9.0, 9.5, 10.0, 10.5, 11.0, 11.5]
fig, axes = plt.subplots(len(MASS_BINS) - 1, 1, figsize=(15, 2.6 * (len(MASS_BINS) - 1)), sharex=True)
for ax, lo, hi in zip(axes, MASS_BINS[:-1], MASS_BINS[1:]):
    in_bin = df['use'] & df['LogMass'].between(lo, hi, inclusive='left')
    for c in CLASSES:
        sel = in_bin & (df['morph_class'] == c)
        if sel.sum() >= 10:
            ax.plot(G, stack(sel, q=(50,))[0], color=CLASS_COLORS[c], lw=0.8, label=f'{c} n={sel.sum()}')
    ax.set_ylabel(f'{lo}–{hi}'); ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('Rest wavelength [Å]')
fig.suptitle('Median spectra in bins of log M* (rows)'); plt.tight_layout(); plt.show()

### 8.5 Small-multiples gallery: random individual galaxies per group

In [ ]:
N_PER_GROUP = 4
rng = np.random.default_rng(RANDOM_STATE)

def smooth(y, k=5):
    """Light running median for display only (keeps NaN gaps)."""
    return pd.Series(y).rolling(k, center=True, min_periods=1).median().where(np.isfinite(y)).to_numpy()

fig, axes = plt.subplots(len(groups_ok), N_PER_GROUP, figsize=(4.5 * N_PER_GROUP, 2.4 * len(groups_ok)),
                         sharex=True, sharey=True)
for r, g in enumerate(groups_ok):
    idx = np.flatnonzero(df['use'] & (df['group'] == g))
    for c, i in enumerate(rng.choice(idx, size=min(N_PER_GROUP, len(idx)), replace=False)):
        ax = axes[r, c]
        ax.plot(G, GROUP_STACKS[g][1], color='grey', lw=0.6)                # group median for reference
        ax.plot(G, smooth(SPEC[i]), color=gcolors[g], lw=0.6)
        ax.set_title(f"{df.at[i, 'name']} | {df.at[i, 'Type']}", fontsize=8)
    axes[r, 0].set_ylabel(g)
axes[0, 0].set_ylim(0, 2.0)
plt.tight_layout(); plt.show()

### 8.6 Interactive exploration
* **Plotly**: median spectrum per Type. Click legend entries to show or hide Types; hover to read values.
* **Widget**: pick a Type and a random seed to browse individual galaxies against the Type median.

In [ ]:
import plotly.graph_objects as go

step = 2                                              # plot every 2nd pixel to keep the figure light
fig = go.Figure()
for t in types_ok:
    fig.add_trace(go.Scattergl(x=G[::step], y=TYPE_STACKS[t][1][::step], mode='lines', name=t,
                               line=dict(width=1, color=mcolors.to_hex(cmap(cnorm(min(TYPE_TTYPE[t], 10)))))))
fig.update_layout(title='Median rest-frame spectrum per Type (click legend to toggle)',
                  xaxis_title='Rest wavelength [Å]', yaxis_title='Normalised flux', height=550)
fig.show()

In [ ]:
import ipywidgets as widgets

def explore(Type, seed):
    """Plot one random good-quality galaxy of `Type` over the Type median."""
    idx = np.flatnonzero(df['use'] & (df['Type'] == Type))
    i = np.random.default_rng(seed).choice(idx)
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.plot(G, TYPE_STACKS[Type][1], color='grey', lw=1, label=f'{Type} median')
    ax.plot(G, smooth(SPEC[i]), color='k', lw=0.6, label=df.at[i, 'name'])
    ax.set(xlim=(G[0], G[-1]), ylim=(0, np.nanpercentile(SPEC[i], 99.5) * 1.2),
           title=f"{df.at[i, 'name']} | {Type} | logM*={df.at[i, 'LogMass']:.2f} | "
                 f"D4000n={df.at[i, 'D4000n']:.2f} | EW(Hα)={df.at[i, 'EW_Ha']:.1f} Å")
    mark_lines(ax); ax.legend(); plt.show()

widgets.interact(explore, Type=widgets.Dropdown(options=types_ok),
                 seed=widgets.IntSlider(value=0, min=0, max=500));

### 8.7 Aperture effect: 1RE vs FoV stacks per class
This runs on a random subsample, because the main matrix only holds `FLUX_1RE`.

In [ ]:
N_SAMPLE = 300
fig, (ax, axr) = plt.subplots(2, 1, figsize=(16, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
for c in CLASSES:
    sub = df[df['use'] & (df['morph_class'] == c)].sample(min(N_SAMPLE, (df['use'] & (df['morph_class'] == c)).sum()),
                                                          random_state=RANDOM_STATE)
    out = Parallel(n_jobs=N_JOBS)(delayed(process_one)(PATH_OF[n], z, 'FLUX_FoV', 'ERROR_FoV')
                                  for n, z in zip(sub['name'], sub['nsa_z']))
    med_fov = np.nanmedian(np.vstack([o[1] for o in out]), axis=0)
    med_1re = np.nanmedian(SPEC[sub.index.to_numpy()], axis=0)
    ax.plot(G, med_1re, color=CLASS_COLORS[c], lw=0.8, label=f'{c} 1RE')
    ax.plot(G, med_fov, color=CLASS_COLORS[c], lw=0.8, ls='--', label=f'{c} FoV')
    axr.plot(G, med_fov / med_1re, color=CLASS_COLORS[c], lw=0.8, label=c)
ax.legend(); ax.set_ylabel('Normalised flux'); mark_lines(ax)
axr.axhline(1, color='grey', ls='--'); axr.set(ylabel='FoV / 1RE', xlabel='Rest wavelength [Å]'); axr.legend()
plt.tight_layout(); plt.show()

### 8.8 (Optional) Images next to spectra with Marvin
[Marvin](https://sdss-marvin.readthedocs.io) can fetch the SDSS image of each MaNGA galaxy (the plate-IFU is the `name` without `manga-`). It needs internet access, and installing it may take a few minutes in Colab.

In [ ]:
# !pip install -q sdss-marvin        # uncomment to install
try:
    from marvin import config
    from marvin.tools.image import Image
    config.setRelease('DR17')

    reps = (df[df['use']].groupby('group', observed=True)
              .apply(lambda g: g.sample(1, random_state=RANDOM_STATE))['name'].tolist())
    fig, axes = plt.subplots(2, len(reps), figsize=(4 * len(reps), 7), gridspec_kw={'height_ratios': [2, 1]})
    for k, name in enumerate(reps):
        i = df.index[df['name'] == name][0]
        axes[0, k].imshow(Image(plateifu=name.replace('manga-', '')).data); axes[0, k].axis('off')
        axes[0, k].set_title(f"{name}\n{df.at[i, 'Type']}", fontsize=9)
        axes[1, k].plot(G, smooth(SPEC[i]), lw=0.5, color='k'); axes[1, k].set_ylim(0, 2)
    plt.tight_layout(); plt.show()
except Exception as e:
    print('Marvin step skipped:', e)

## 9. Save outputs

In [ ]:
OUT_TABLE = os.path.join(CACHE_DIR, 'galaxy_eda_table.parquet')
df.to_parquet(OUT_TABLE)                                         # metadata + QC + indices + PCs

np.savez_compressed(os.path.join(CACHE_DIR, 'stacks_by_type.npz'), wave=G,
                    **{f'type_{t}': p for t, p in TYPE_STACKS.items()},
                    **{f'class_{c}': p for c, p in CLASS_STACKS.items()})
sep.to_csv(os.path.join(CACHE_DIR, 'early_late_separation.csv'))
print('Saved to', CACHE_DIR)